# 02 Resolve 与 Get

在 01 写入的种子图上检验两个读取算子：

- **`Resolve`**：把一个说法解析为已存对象的引用；
- **`Get`**：按引用装配对象视图。

契约见 `intents_decompose.md` §4.1 和 `graph_model_v2.md` §2.4。实现在 `operators/resolve.py`、`operators/get.py`，共用部件在 `utils/`（`fusion`、`embedding`）。

除了建索引、给对象补算向量（检索用的派生属性），本 notebook 不写图。用例全部取自库里的真实种子，不造数据。走不到的路径在最后列出。

## 1. 准备：索引与向量

- **全文索引**（`utils/schema.py` 的 `FULLTEXT`）：
  - `namekey_raw`：NameKey 的 `raw`，默认分词。名称和 alias 都在 NameKey 上，对象上没有 aliases 列表，所以名称词面通道查的是 NameKey。
  - `entity_texts`：Entity 的 `description`，english 分词。
  - `concept_texts`：Concept 的 `definition`、`scope_note`，english 分词。
- **向量**：Qwen3-Embedding-8B，与 e08 共用同一个 vLLM 服务，4096 维，cosine 相似度。
  - 每个对象的向量输入按类别固定：名称、由 NameKey 拼出的 alias，加上该类的文本字段。
  - 写入侧不加任务说明，查询侧加。
  - 每个对象另存 `embedding_key`（模型名 + 输入文本的哈希）。`sync_embeddings()` 只补算缺失或过期的向量，所以重跑没有副作用。可复现的入口是 `make embed`。

In [ ]:
import pandas as pd

from e09.env_check import check_graph
from e09.utils.embedding import EMBED_MODEL, EMBED_URL, ensure_vector_indexes, sync_embeddings
from e09.utils.graph import ensure_schema, q

assert not check_graph(), "不是 v2 库，停止"
ensure_schema()
ensure_vector_indexes()
print(EMBED_MODEL, EMBED_URL)
print("补算向量", sync_embeddings(), "个对象；重跑补算", sync_embeddings(), "个")
display(pd.DataFrame(q("SHOW INDEXES YIELD name, type, labelsOrTypes, properties, state "
                       "WHERE type IN ['FULLTEXT', 'VECTOR'] RETURN name, type, labelsOrTypes, properties, state")))

## 2. Get：对象视图

`get(refs)` 只装配对象本身：主 Label、kind、属性、aliases、identifiers。关系展开是 Context 和 Search 的职责，这里不做。

- aliases 不存在对象上，读取时由 NameKey 拼出：`raw` 与对象 `name` 不同的键就是 alias；
- `name_keys` 保留每个键的原字符串、键和状态，供核对；
- 库里没有的 ref 放进 `missing`，不报错。

In [ ]:
from e09.operators.get import get

g = get(["dataset_0001", "task_0002", "metric_0001", "code_0001"])
print("missing:", g["missing"])
pd.DataFrame([{k: v for k, v in item.items() if k != "name_keys"} for item in g["items"]])

In [ ]:
g["items"][0]["name_keys"]   # ETT：一个 name、两个 alias

## 3. Resolve：契约与展示

```python
resolve(query={identifier?, mention?, text?}, *, kind, scope="global", mode="read")
  -> {stage, status, refs, match_trace, states, coverage, execution}
```

参数说明：

- query 的三个分量：
  - `identifier` 是 `"<namespace>:<value>"`；
  - `mention` 是说法，直接传一个字符串即表示 mention；
  - `text` 是可选的说明文字，与对象的 description / definition 同类，只进入语义阶段；
- type 由 kind 推出，一次调用只查一个 kind。

**三级逐级解析：**

| 级 | 依据 | 结果 |
| --- | --- | --- |
| **id** | identifiers 精确匹配 | 唯一命名空间单个命中 → resolved。非唯一命名空间（url）的命中只缩小候选：多个命中为 ambiguous，单个命中为 candidates |
| **alias** | name-key-v1 精确键 | active 键唯一命中 → resolved；已隔离为 ambiguous 的键 → ambiguous |
| **semantic** | 名称词面、文本、向量三路，按 RRF 融合 | candidates，value=U，交外部 A_pred 确认 |

**id 级与 alias 级同时有结果时取交集。** 交集唯一则 resolved。仍不唯一或为空则保留 ambiguous；名称命中了某个对象、交集却为空时，`resolution` 记为 conflicting。

**两种模式：**
- **read：** 前两级 resolved 即停止；ambiguous 时仍跑语义阶段，结果记入 `match_trace`。
- **write：** 精确命中后仍跑语义查重；非唯一标识的命中只作查重候选（candidates）。

**返回字段：**
- `states` 沿用 `intents_decompose.md` §4.3.2.1 的统一状态维度：
  - `access`：matched / empty / unprocessed；
  - `resolution`：resolved / ambiguous / conflicting / unresolved / missing；missing 时带 `missing_in=store`；
  - `value`：T / U；
  - 另附 `issues`（不一致的明细）和 `errors`。
- `execution` 为 ok / partial / error。通道执行失败、又因此没有引用时，`status=unprocessed`，不当作 none。
- `coverage` 记录 kind、scope、各通道状态（ok / disabled / skipped / error）、召回规模、向量的过滤方式，以及是否截断。

`show` 是浏览用的视图：给每个 ref 补上名称，`match_trace` 逐级列出。

In [ ]:
from e09.operators.resolve import resolve

U_ETT = "url:https://github.com/zhouhaoyi/ETDataset"


def names(refs):
    return {i["ref"]: i["name"] for i in get(refs)["items"]}


def show(r):
    nm = names(r["refs"] + [c["id"] for s in r["match_trace"] for c in s.get("candidates", [])]
                + [x for s in r["match_trace"] for x in s.get("hits", [])])
    print(f"stage={r['stage']}  status={r['status']}  refs={[f'{x} {nm.get(x, x)}' for x in r['refs']]}")
    print(f"states: access={r['states']['access']} resolution={r['states']['resolution']} "
          f"value={r['states']['value']} execution={r['execution']}")
    for i in r["states"]["issues"]:
        print("  不一致：", i)
    for e in r["states"]["errors"]:
        print("  错误：", e)
    print("channels:", r["coverage"]["channels"], " truncated:", r["coverage"]["truncated"])
    for s in r["match_trace"]:
        if s["stage"] == "semantic":
            print(f"  trace semantic（{s['role']}）：")
            display(pd.DataFrame([{"id": c["id"], "name": nm.get(c["id"]), "rrf": c["rrf"],
                                   "evidence": "; ".join(f"{ch} #{e['rank']} {e['score']}" for ch, e in c["evidence"].items())}
                                  for c in s["candidates"]]))
        else:
            print("  trace", {k: ([f"{x} {nm.get(x, x)}" for x in v] if k == "hits" else v) for k, v in s.items()})
    return r

## 4. 用例

### 4.1 非唯一标识的多重命中

`url:https://github.com/zhouhaoyi/ETDataset` 由 ETT 与 ETTh1–ETTm2 共用。

**只给标识：** 结果为 `stage=id, status=ambiguous`，返回全部 5 个命中，不会静默落到下一级。没有 mention 也没有 text，语义阶段的三个通道全部关闭。

In [ ]:
r = show(resolve({"identifier": U_ETT}, kind="Dataset"))
assert (r["stage"], r["status"], len(r["refs"])) == ("id", "ambiguous", 5)

**标识加名称 `ETTh1`：** 名称唯一命中 ETTh1，且 ETTh1 在 5 个标识命中之中。交集唯一，结果为 resolved，`match_trace` 记录了两级依据。read 模式下 resolved 后不再跑语义阶段。

In [ ]:
r = show(resolve({"identifier": U_ETT, "mention": "ETTh1"}, kind="Dataset"))
assert (r["stage"], r["status"]) == ("alias", "resolved") and names(r["refs"]) == {r["refs"][0]: "ETTh1"}

**标识加名称 `Weather`：** Weather 是另一个仓库下的数据集。名称唯一命中 Weather，但它不在标识的 5 个命中里，交集为空。结果保留 ambiguous（refs 仍是标识的 5 个命中），`resolution` 记为 conflicting，`issues` 给出 `id-name-disjoint` 的明细：标识和名称指向了不同对象，需要外部判断哪一边写错了。

In [ ]:
r = show(resolve({"identifier": U_ETT, "mention": "Weather"}, kind="Dataset"))
assert r["status"] == "ambiguous" and r["states"]["resolution"] == "conflicting"
assert [i["rule"] for i in r["states"]["issues"]] == ["id-name-disjoint"]

**写入模式下同一个 url：** 多重命中只作查重候选，结果为 candidates，不判为 ambiguous，也不直接复用。新对象是 5 个之一，还是同一仓库里的另一个数据集，由外部判断。

In [ ]:
r = show(resolve({"identifier": U_ETT}, kind="Dataset", mode="write"))
assert (r["stage"], r["status"], len(r["refs"])) == ("id", "candidates", 5)

### 4.2 alias 级

**`LTSF`（Task）：** 是 `Long-term time series forecasting` 的注册 alias，唯一命中，`stage=alias, resolved`。

In [ ]:
r = show(resolve("LTSF", kind="Task"))
assert (r["stage"], r["status"]) == ("alias", "resolved")
assert names(r["refs"]) == {r["refs"][0]: "Long-term time series forecasting"}

**`ltsf`：** name-key-v1 保留大小写，`ltsf|Task|global` 没有注册，前两级都不命中，进入语义阶段。LTSF 排在第一，但只是 candidates（value=U），需要外部确认。确认之后可以把 `ltsf` 注册为 alias，供以后复用（`intents_decompose.md` §4.1："新说法经确认后，通过独立显式写入注册 alias"）。

In [ ]:
r = show(resolve("ltsf", kind="Task"))
assert (r["stage"], r["status"], r["states"]["value"]) == ("semantic", "candidates", "U")
assert names(r["refs"][:1]) == {r["refs"][0]: "Long-term time series forecasting"}

**`MSE` 按 Method 查：** `MSE` 只注册为 Metric 的键，kind 不同就是不同的键，前两级都不命中。语义阶段也只在 Method 内召回，不会跨 kind 返回 Metric 的 MSE。返回的几个方法与 MSE 无关，但仍是 candidates：语义阶段没有阈值，只要该 kind 有对象，向量通道总会给出近邻。所以 candidates 本身不能说明"找到了"，这一点由外部确认来承担。

In [ ]:
r = show(resolve("MSE", kind="Method"))
assert r["stage"] == "semantic" and all(i["kind"] == "Method" for i in get(r["refs"])["items"])

### 4.3 写入模式：精确命中后仍查重

**以 write 模式查 `MSE`（Metric）：** 精确键命中，结果是 resolved。语义查重仍然执行，近邻（不含已命中的对象）记入 `match_trace`，由外部判断其中有没有重复对象，或者需要补哪些 alias。

In [ ]:
r = show(resolve("MSE", kind="Metric", mode="write"))
assert (r["stage"], r["status"]) == ("alias", "resolved")
assert r["match_trace"][-1]["role"] == "dedup" and r["refs"][0] not in [c["id"] for c in r["match_trace"][-1]["candidates"]]

### 4.4 无命中

**`FlashAttention` 按 Code 查：** 图里还没有 Code 对象，三级都没有引用，结果为 `stage=semantic, status=none, access=empty, resolution=missing, missing_in=store`。在有语义通道可用时，这是唯一会出现 none 的情况。

In [ ]:
r = show(resolve("FlashAttention", kind="Code"))
assert (r["status"], r["states"]["access"], r["states"]["missing_in"]) == ("none", "empty", "store")

**`Quantum annealing scheduler` 按 Dataset 查：** 库里显然没有这样的数据集，但 Dataset 有 20 个对象，向量通道仍会返回最近的几个，结果是 candidates，不是 none。这和 4.2 中 `MSE` 按 Method 查是同一个现象：

- "三级均未返回引用才是 none"的契约本身成立；
- 但只要有向量通道，"没有"就几乎不会以 none 的形式出现，而要由外部对 candidates 作出否定判断。

这一点要计入 Resolve 的确认成本（主张 A）。是否需要给语义候选加阈值或者"可能不存在"的提示，等论文入库后，按真实查询的情况再讨论。

In [ ]:
r = show(resolve("Quantum annealing scheduler", kind="Dataset"))
assert r["status"] == "candidates"

### 4.5 带说明文字

给出 `text` 时会多开一个文本通道，向量的查询文本也换成 `mention: text`。这里查 `IMS`：它本身是 `Iterated multi-step forecasting` 的注册 alias，所以 read 模式直接 resolved；换成 write 模式，可以看到三路通道各自召回了什么。

In [ ]:
r = show(resolve({"mention": "IMS", "text": "apply a one-step forecaster recursively, feeding predictions back as inputs"},
                 kind="Method", mode="write"))
assert r["status"] == "resolved" and all(s.startswith("ok") for s in r["coverage"]["channels"].values())

## 5. 还没覆盖的

- **唯一命名空间的命中与冲突**（`arxiv`、`doi`、`s2`）：种子里没有这类标识，id 级的 resolved 与 `unique-identifier-duplicated` 两条路径要等论文入库后才会走到。
- **已隔离为 ambiguous 的键**：只在批量导入或事后核查发现冲突时产生，种子入库不会出现。
- **Split 的作用域**：切分的 scope 是父数据集，种子里没有 Split，所以 `scope` 参数目前只测了 global。
- **Content**：Content 不走 alias，只支持已注册标识与内容候选通道。等 Experiment 等对象入库后再加。
- **向量先取后过滤**：向量索引先取 $\text{POOL} \times 5 = 100$ 个，再按 kind 过滤。kind 内的对象一多就可能漏掉可行候选，`coverage.vector` 记录了这一做法。
- **语义阶段的质量**：三路的召回与排序质量没有系统测量。等论文入库后，用真实的 mention 与外部确认结果来测。